# Día 5 · Unidad 5 — POO aplicada (III): dataclasses, Enum y (opcional) magic methods avanzados

**Objetivos de aprendizaje**
- Usar `@dataclass` para clases centradas en datos, evitando escribir `__init__`/`__repr__`/`__eq__` a mano.
- Evitar el error clásico de los valores por defecto mutables en una dataclass.
- Usar `Enum` para restringir los valores válidos de un atributo.
- (Opcional, solo si sobra tiempo) asomarse a magic methods avanzados y a la idea general de patrón de diseño.

**Duración estimada:** 45-60 minutos (sin contar el bloque opcional).

**Requisitos previos:** `02_teoria_properties_herencia.ipynb` de este mismo día.

## 1. `dataclasses`: clases centradas en datos

Muchas clases existen solo para agrupar datos, sin apenas comportamiento propio — como una fila de una tabla de pólizas. Para esos casos, escribir `__init__`, `__repr__` y `__eq__` a mano es trabajo repetitivo. El decorador `@dataclass` (desde Python 3.7) los genera automáticamente a partir de las anotaciones de tipo.

In [1]:
from dataclasses import dataclass


@dataclass
class Asegurado:
    nombre: str
    dni: str
    edad: int
    activo: bool = True   # valor por defecto, igual que en una función


asegurado_1 = Asegurado(nombre="Anna Puig", dni="12345678A", edad=41)
asegurado_2 = Asegurado(nombre="Anna Puig", dni="12345678A", edad=41)

print(asegurado_1)                  # __repr__ generado automáticamente, con todos los campos
print(asegurado_1 == asegurado_2)   # __eq__ generado automáticamente: compara los valores, no la identidad

Asegurado(nombre='Anna Puig', dni='12345678A', edad=41, activo=True)
True


Con una clase normal habrías tenido que escribir tú mismo `__init__`, `__repr__` y `__eq__` para conseguir exactamente este comportamiento — es el mismo patrón que viste en el notebook 1, pero generado por Python en vez de a mano.

> ⚠️ **Error típico**: usar una lista (u otro tipo mutable) como valor por defecto directamente, igual que con los atributos de clase mutables del notebook 1. En una `dataclass` esto ni siquiera se puede escribir así — Python lo bloquea con un error explícito en cuanto defines la clase:

In [2]:
try:
    @dataclass
    class CarteraConBug:
        titular: str
        polizas: list[str] = []   # ¡mal! Python lo rechaza al definir la clase
except ValueError as e:
    print("Error esperado:", e)

Error esperado: mutable default <class 'list'> for field polizas is not allowed: use default_factory


La forma correcta es `field(default_factory=list)`, que le dice a Python "crea una lista **nueva** para cada instancia", exactamente igual que hacíamos a mano con `coberturas if coberturas is not None else []` en el notebook 1.

In [3]:
from dataclasses import dataclass, field


@dataclass
class Cartera:
    titular: str
    polizas: list[str] = field(default_factory=list)


cartera_a = Cartera(titular="Anna Puig")
cartera_b = Cartera(titular="Marc Soler")

cartera_a.polizas.append("POL-001")

print("Pólizas de Anna:", cartera_a.polizas)
print("Pólizas de Marc:", cartera_b.polizas)   # sigue vacía: cada instancia tiene su propia lista

Pólizas de Anna: ['POL-001']
Pólizas de Marc: []


**Para ti:** crea una `dataclass` `Siniestro` con campos `numero_poliza: str`, `importe: float` y `descripcion: str = ""`. Crea dos siniestros con los mismos valores y comprueba que `==` los considera iguales (algo que, con una clase normal sin `__eq__`, habría dado `False`).

In [4]:
# Tu código aquí

## 2. Combinar dataclasses con `Enum`

Cuando un atributo solo puede tomar un conjunto cerrado de valores (por ejemplo, el tipo de cobertura), un `str` suelto es frágil: nada impide escribir `"responsabilidad civil"` en una póliza y `"RC"` en otra. `Enum` fija de una vez las opciones válidas.

In [5]:
from enum import Enum, auto


class TipoCobertura(Enum):
    RESPONSABILIDAD_CIVIL = auto()
    DANOS_PROPIOS = auto()
    ROBO = auto()


@dataclass
class Poliza:
    asegurado: str
    prima: float
    coberturas: list[TipoCobertura] = field(default_factory=list)
    vigente: bool = True


poliza = Poliza(
    asegurado="Anna Puig",
    prima=480.0,
    coberturas=[TipoCobertura.RESPONSABILIDAD_CIVIL, TipoCobertura.DANOS_PROPIOS],
)

print(poliza)
print(TipoCobertura.RESPONSABILIDAD_CIVIL)

Poliza(asegurado='Anna Puig', prima=480.0, coberturas=[<TipoCobertura.RESPONSABILIDAD_CIVIL: 1>, <TipoCobertura.DANOS_PROPIOS: 2>], vigente=True)
TipoCobertura.RESPONSABILIDAD_CIVIL


Ahora un editor de código (o Claude Code) puede autocompletar las coberturas válidas, y `TipoCobertura.ROBO` es un error de escritura si no está en la lista de valores del `Enum` — mientras que `"robo"` como texto suelto no habría avisado de nada si alguien lo escribe mal.

## Resumen

- `@dataclass` genera `__init__`, `__repr__` y `__eq__` a partir de las anotaciones de tipo — útil para clases centradas en datos.
- Usar una lista/diccionario como valor por defecto directamente en una `dataclass` da error al definir la clase; la solución es `field(default_factory=list)`, igual que en `__init__` se inicializa dentro del cuerpo en vez de como atributo de clase.
- `Enum` restringe los valores válidos de un atributo a un conjunto cerrado y conocido, evitando strings sueltos con errores de escritura.
- `dataclass` y `Enum` combinan bien: una dataclass puede tener un campo tipado como un `Enum`.
- No sustituyen a una clase con lógica propia: si necesitas properties calculadas, validación en el `__init__` o herencia con comportamiento, una `dataclass` sigue siendo una clase normal por debajo (puedes añadirle métodos), pero para ese caso a veces conviene simplemente una clase escrita a mano, como en los notebooks 1 y 2.

## 3. (Opcional, solo si sobra tiempo) Magic methods avanzados y patrones de diseño

Este bloque es **bonus**: no es obligatorio para el programa del día. Si vais bien de tiempo, aquí tenéis una probadita de dos temas que se pueden ampliar por vuestra cuenta más adelante.

### Magic methods avanzados

Además de `__repr__`, `__eq__`, `__lt__` y `__len__` (notebook 1), Python permite personalizar muchos más operadores. Dos ejemplos con una `Cartera` de pólizas:

In [6]:
@dataclass
class PolizaSimple:
    asegurado: str
    prima: float


class Cartera:
    def __init__(self, titular: str):
        self.titular = titular
        self._polizas: list[PolizaSimple] = []

    def __add__(self, poliza: PolizaSimple) -> "Cartera":
        """Permite escribir `cartera + poliza` para añadir una póliza."""
        self._polizas.append(poliza)
        return self

    def __contains__(self, asegurado: str) -> bool:
        """Permite escribir `"Anna Puig" in cartera`."""
        return any(p.asegurado == asegurado for p in self._polizas)

    def __iter__(self):
        """Permite recorrer la cartera con un `for poliza in cartera:`."""
        return iter(self._polizas)


cartera = Cartera(titular="Bróker Central")
cartera = cartera + PolizaSimple(asegurado="Anna Puig", prima=480.0)
cartera = cartera + PolizaSimple(asegurado="Marc Soler", prima=320.0)

print("Anna Puig" in cartera)
for p in cartera:
    print(p)

True
PolizaSimple(asegurado='Anna Puig', prima=480.0)
PolizaSimple(asegurado='Marc Soler', prima=320.0)


### Una idea general de "patrón de diseño"

Un patrón de diseño es una solución con nombre a un problema de diseño que aparece una y otra vez — no es código que se copia y pega, sino una forma de estructurar la solución. Ya usasteis uno sin llamarlo así: el método de clase `Poliza.nueva(...)` del notebook 1 es una versión simplificada del patrón **Factory** (fábrica): en vez de que quien usa la clase construya el objeto directamente, se lo pide a un método que sabe cómo hacerlo bien (en ese caso, asignando el número de póliza automáticamente).

No hace falta memorizar catálogos de patrones para este curso — basta con reconocer que, si una misma necesidad de diseño se repite mucho, probablemente ya tiene nombre y alguien ya lo resolvió antes. Buscar "factory pattern python" o "strategy pattern python" es un buen punto de partida si en el futuro os encontráis reinventando la misma solución varias veces.

**Siguiente:** `04_ejercicios.ipynb`, para practicar todo el día — incluido un segundo ejercicio de "encuentra el fallo sutil", distinto al de este notebook.